In [ ]:
from dune_client.client import DuneClient
import pandas as pd
from datetime import datetime

# === SETUP ===
API_KEY = "7Nm1PkuQZ2TX8gaAwt1eyjilFXGKbfRw"
dune = DuneClient(api_key=API_KEY)

def run_dune_query(sql: str, filename: str):
    print(f"Running query for {filename}...")
    print("⚠️ This may take several minutes (7 years of Ethereum data)...")
    
    results = dune.run_sql(query_sql=sql)
    df = pd.DataFrame(results.result.rows) # type: ignore
    df.to_csv(filename, index=False)
    
    print(f"✅ Saved {len(df):,} rows to {filename}")
    print(f"   Date range: {df['day'].min()} → {df['day'].max()}")
    return df


# ==================== ETH 7-YEAR QUERY ====================
eth_sql = """
SELECT 
    DATE_TRUNC('day', block_time) AS day,
    COUNT(*) AS tx_count,
    COUNT(DISTINCT "from") AS active_senders,
    COUNT(DISTINCT "to") AS active_receivers,
    SUM(gas_used) AS total_gas_used,
    SUM(value) / 1e18 AS total_eth_transferred
FROM ethereum.transactions
WHERE block_time >= TIMESTAMP '2019-01-01 00:00:00'
  AND block_time <= TIMESTAMP '2025-12-31 23:59:59'
GROUP BY 1
ORDER BY 1 ASC
"""

df_eth = run_dune_query(eth_sql, "dune_eth_daily_onchain_7years.csv")



Running query for dune_eth_daily_onchain_7years.csv...
⚠️ This may take several minutes (7 years of Ethereum data)...
✅ Saved 2,557 rows to dune_eth_daily_onchain_7years.csv
   Date range: 2019-01-01 00:00:00.000 UTC → 2025-12-31 00:00:00.000 UTC


In [ ]:
# # XRPL query (90 days)
# xrpl_sql = """
# SELECT 
#     ledger_close_date AS day,
#     COUNT(*) AS tx_count,
#     COUNT(DISTINCT account) AS active_addresses,
#     SUM(CAST(amount.value AS DOUBLE)) / 1e6 AS total_xrp_moved,
#     AVG(CAST(fee AS DOUBLE)) / 1e6 AS avg_fee_xrp
# FROM xrpl.transactions
# WHERE ledger_close_date >= CURRENT_DATE - INTERVAL '1825' DAY
# GROUP BY 1
# ORDER BY 1 DESC
# """
# df_xrpl = run_dune_query(xrpl_sql, "dune_xrpl_daily_onchain_1825d.csv")

# print("\n🎉 Both datasets ready for your ML model!")

Running query for dune_xrpl_daily_onchain_90d.csv...
✅ Saved 91 rows to dune_xrpl_daily_onchain_90d.csv

🎉 Both datasets ready for your ML model!


In [2]:
import ccxt
import pandas as pd
from datetime import datetime, timedelta
import time

exchange = ccxt.coinbase()   # or ccxt.coinbasepro()

symbol = 'ETH/USD'
timeframe = '1m'
days = 1825

since = int((datetime.utcnow() - timedelta(days=days)).timestamp() * 1000)

print(f"Fetching {days} days of 1m ETH/USD using ccxt...")

all_candles = []
while since < int(datetime.utcnow().timestamp() * 1000):
    candles = exchange.fetch_ohlcv(symbol, timeframe, since=since, limit=1000)
    if not candles:
        break
    all_candles.extend(candles)
    since = candles[-1][0] + 1  # move to next candle
    print(f"  → Fetched {len(all_candles):,} candles so far...")
    time.sleep(0.3)  # still respectful

df = pd.DataFrame(all_candles, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
df = df.drop_duplicates(subset='timestamp').sort_values('timestamp').reset_index(drop=True)

df.to_csv("eth_usd_1min_ccxt.csv", index=False)
print(f"\n✅ Done! {len(df):,} rows saved.")

Fetching 1825 days of 1m ETH/USD using ccxt...
  → Fetched 300 candles so far...
  → Fetched 600 candles so far...
  → Fetched 900 candles so far...
  → Fetched 1,200 candles so far...
  → Fetched 1,500 candles so far...
  → Fetched 1,800 candles so far...
  → Fetched 2,100 candles so far...
  → Fetched 2,400 candles so far...
  → Fetched 2,700 candles so far...
  → Fetched 3,000 candles so far...
  → Fetched 3,300 candles so far...
  → Fetched 3,600 candles so far...
  → Fetched 3,900 candles so far...
  → Fetched 4,200 candles so far...
  → Fetched 4,500 candles so far...
  → Fetched 4,800 candles so far...
  → Fetched 5,100 candles so far...
  → Fetched 5,400 candles so far...
  → Fetched 5,700 candles so far...
  → Fetched 6,000 candles so far...
  → Fetched 6,300 candles so far...
  → Fetched 6,600 candles so far...
  → Fetched 6,900 candles so far...
  → Fetched 7,200 candles so far...
  → Fetched 7,500 candles so far...
  → Fetched 7,800 candles so far...
  → Fetched 8,100 candl

KeyboardInterrupt: 

In [8]:
import requests
import zipfile
import pandas as pd
import os
from datetime import datetime
from tqdm import tqdm
import yfinance as yf

# ========================= CONFIG =========================
SYMBOL = "ETHUSDT"
INTERVAL = "1m"
START_YEAR = 2019
OUTPUT_CSV = "eth_usd_1min_adjusted.csv"
DATA_DIR = "binance_data"

os.makedirs(DATA_DIR, exist_ok=True)
# =========================================================

def download_monthly_data(year, month):
    url = f"https://data.binance.vision/data/spot/monthly/klines/{SYMBOL}/{INTERVAL}/{SYMBOL}-{INTERVAL}-{year:04d}-{month:02d}.zip"
    filename = os.path.join(DATA_DIR, f"{SYMBOL}-{INTERVAL}-{year:04d}-{month:02d}.zip")
    
    if os.path.exists(filename):
        return filename
    
    print(f"Downloading {year}-{month:02d}...")
    try:
        resp = requests.get(url, stream=True, timeout=30)
        resp.raise_for_status()
        with open(filename, 'wb') as f:
            for chunk in tqdm(resp.iter_content(chunk_size=1024*1024), desc=f"{year}-{month:02d}", unit="MB", leave=False):
                f.write(chunk)
        return filename
    except Exception as e:
        if "404" in str(e):
            return None  # Future month - expected
        print(f"Failed {year}-{month:02d}: {e}")
        return None


# ============== 1. DOWNLOAD (Safe up to last month) ==============
print("=== Downloading Binance ETHUSDT 1m data ===")
all_files = []
current_year = datetime.now().year
current_month = datetime.now().month

for year in range(START_YEAR, current_year + 1):
    end_month = current_month - 1 if year == current_year else 12   # Skip current incomplete month
    for month in range(1, end_month + 1):
        zip_file = download_monthly_data(year, month)
        if zip_file:
            all_files.append(zip_file)

print(f"\nFound/Downloaded {len(all_files)} monthly files.")

# ============== 2. EXTRACT + COMBINE (with error handling) ==============
print("Extracting and combining...")
all_dfs = []

for zip_path in tqdm(all_files, desc="Processing ZIPs"):
    try:
        with zipfile.ZipFile(zip_path) as z:
            csv_name = z.namelist()[0]
            with z.open(csv_name) as f:
                # Read with low_memory and handle bad lines
                df = pd.read_csv(f, header=None, low_memory=False, on_bad_lines='skip')
                
                df.columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume',
                              'close_time', 'quote_volume', 'trades', 'taker_base', 
                              'taker_quote', 'ignore']
                
                # Safe timestamp conversion
                df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
                
                # Drop rows with invalid timestamps
                df = df.dropna(subset=['timestamp'])
                
                df = df[['timestamp', 'open', 'high', 'low', 'close', 'volume']]
                all_dfs.append(df)
    except Exception as e:
        print(f"Error processing {zip_path}: {e}")

final_df = pd.concat(all_dfs, ignore_index=True)
final_df = final_df.drop_duplicates(subset='timestamp').sort_values('timestamp').reset_index(drop=True)

print(f"Combined raw data: {len(final_df):,} rows")

# ============== 3. USDT → USD ADJUSTMENT ==============
print("Downloading daily USD/USDT rates...")

usdt = yf.download("USDT-USD", 
                   start=final_df['timestamp'].min().date(), 
                   end=datetime.now().date() + pd.Timedelta(days=1), 
                   progress=False)

usdt = usdt[['Close']]
usdt.columns = ['usd_per_usdt']
usdt.index = pd.to_datetime(usdt.index).date

final_df['date'] = final_df['timestamp'].dt.date
final_df = final_df.merge(usdt, left_on='date', right_index=True, how='left')

# Adjust prices
for col in ['open', 'high', 'low', 'close']:
    final_df[col] = final_df[col] * final_df['usd_per_usdt']

final_df = final_df[['timestamp', 'open', 'high', 'low', 'close', 'volume']]
final_df = final_df.sort_values('timestamp').reset_index(drop=True)

# ============== 4. SAVE ==============
final_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n🎉 SUCCESS!")
print(f"   → Saved {len(final_df):,} clean rows to {OUTPUT_CSV}")
print(f"   → Date range: {final_df['timestamp'].min()} → {final_df['timestamp'].max()}")
print(final_df.head())

=== Downloading Binance ETHUSDT 1m data ===

Found/Downloaded 86 monthly files.
Extracting and combining...


Processing ZIPs: 100%|██████████| 86/86 [00:06<00:00, 13.05it/s]


Combined raw data: 3,152,390 rows

🎉 SUCCESS!
   → Saved 3,152,390 clean rows to eth_usd_1min_adjusted.csv
   → Date range: 2019-01-01 00:00:00 → 2024-12-31 23:59:00
            timestamp        open        high         low       close  \
0 2019-01-01 00:00:00  133.860134  133.951784  133.829584  133.860134   
1 2019-01-01 00:01:00  133.849950  133.849950  133.412066  133.625917   
2 2019-01-01 00:02:00  133.625917  133.676833  133.595367  133.646283   
3 2019-01-01 00:03:00  133.646283  133.707384  133.636100  133.687017   
4 2019-01-01 00:04:00  133.676833  133.717567  133.646283  133.676833   

      volume  
0  240.20353  
1  621.28854  
2   64.85972  
3  151.86484  
4  190.91042  


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

# ========================= CONFIG =========================
PRICE_CSV = "eth_usd_1min_adjusted.csv"      # Change to full file when ready
ONCHAIN_CSV = "dune_eth_daily_onchain_7years.csv"
OUTPUT_CSV = "eth_usd_1min_with_7y_onchain_features.csv"
# =========================================================

print("Loading price data...")
price_df = pd.read_csv(PRICE_CSV, parse_dates=["timestamp"])
price_df = price_df.set_index("timestamp").sort_index()

print("Loading on-chain data...")
onchain_df = pd.read_csv(ONCHAIN_CSV, parse_dates=["day"])
onchain_df = onchain_df.rename(columns={"day": "timestamp"})
onchain_df = onchain_df.set_index("timestamp").sort_index()

print(f"Price rows: {len(price_df):,}")
print(f"On-chain days: {len(onchain_df):,}")

# ============== MERGE: Resample daily on-chain to 1-minute ==============
print("Merging daily on-chain → 1-minute (forward-fill)...")
onchain_resampled = onchain_df.resample('1T').ffill()

merged = price_df.join(onchain_resampled, how="left")

print(f"Final merged rows: {len(merged):,}")

# ============== FEATURE ENGINEERING ==============
print("Engineering features...")

df = merged.copy()
price = df["close"]

# 1. Basic returns & lags
df["return"] = np.log(price / price.shift(1))
for lag in [1, 5, 15, 30, 60, 240, 1440]:  # up to 1 day
    df[f"return_lag_{lag}"] = df["return"].shift(lag)
    df[f"price_lag_{lag}"] = price.shift(lag)

# 2. Rolling statistics
for window in [5, 10, 20, 60, 180, 1440]:  # 5min → 1day
    df[f"rolling_mean_{window}"] = price.rolling(window).mean()
    df[f"rolling_std_{window}"] = price.rolling(window).std()
    df[f"rolling_vol_{window}"] = df["return"].rolling(window).std()

# 3. Technical indicators
# RSI (14)
delta = price.diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = -delta.where(delta < 0, 0).rolling(14).mean()
rs = gain / loss
df["rsi_14"] = 100 - (100 / (1 + rs))

# MACD
df["ema_12"] = price.ewm(span=12, adjust=False).mean()
df["ema_26"] = price.ewm(span=26, adjust=False).mean()
df["macd"] = df["ema_12"] - df["ema_26"]
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

# Bollinger Bands
df["bb_middle"] = price.rolling(20).mean()
df["bb_std"] = price.rolling(20).std()
df["bb_upper"] = df["bb_middle"] + 2 * df["bb_std"]
df["bb_lower"] = df["bb_middle"] - 2 * df["bb_std"]

# 4. On-chain specific features (very powerful)
df["tx_count"] = df["tx_count"]
df["active_senders"] = df["active_senders"]
df["active_receivers"] = df["active_receivers"]
df["total_eth_transferred"] = df["total_eth_transferred"]
df["total_gas_used"] = df["total_gas_used"]

# Lagged & moving averages of on-chain metrics
for col in ["tx_count", "active_senders", "active_receivers", "total_eth_transferred"]:
    df[f"{col}_lag1"] = df[col].shift(1440)          # 1-day lag
    df[f"{col}_ma7"] = df[col].rolling(7*1440).mean()  # 7-day MA

# Interaction features
df["price_vs_tx"] = price / (df["tx_count"] + 1)
df["price_vs_active_senders"] = price / (df["active_senders"] + 1)

# 5. Time features
df["hour"] = df.index.hour # type: ignore
df["dow"] = df.index.dayofweek # type: ignore
df["month"] = df.index.month # pyright: ignore[reportAttributeAccessIssue]

# ============== TARGET VARIABLES ==============
df["target_next_close"] = price.shift(-1)                    # regression
df["target_direction"] = (df["target_next_close"] > price).astype(int)  # classification

# Drop rows with NaN (first few rows + any gaps)
df = df.dropna()

print(f"✅ Final dataset after feature engineering: {len(df):,} rows × {len(df.columns)} columns")

# ============== SAVE ==============
df.to_csv(OUTPUT_CSV)
print(f"\n🎉 Saved to {OUTPUT_CSV}")
print(df.head())

Loading price data...
Loading on-chain data...
Price rows: 3,152,390
On-chain days: 2,557
Merging daily on-chain → 1-minute (forward-fill)...


TypeError: Cannot join tz-naive with tz-aware DatetimeIndex

In [1]:
import polars as pl

# ========================= CONFIG =========================
FULL_FILE = "eth_usd_1min_with_7y_onchain_features.csv"
LITE_FILE = "eth_usd_1min_with_7y_onchain_features_lite.csv"
ROWS_TO_KEEP = 700_000
# =========================================================

print(f"Scanning the first {ROWS_TO_KEEP:,} rows using Polars...")

# Polars lazily scans, takes only the rows we need, then collects it into memory
df = pl.scan_csv(FULL_FILE).head(ROWS_TO_KEEP).collect()

# Save using Polars' native CSV writer
df.write_csv(LITE_FILE)

print(f"🎉 Lite file created: {LITE_FILE}")
print(f"   Rows: {df.height:,}") # Polars uses .height for row count

if 'timestamp' in df.columns:
    print(f"   Date range: {df['timestamp'][0]} → {df['timestamp'][-1]}")

print("\nFirst 5 rows preview:")
print(df.head(5))

Scanning the first 700,000 rows using Polars...
🎉 Lite file created: eth_usd_1min_with_7y_onchain_features_lite.csv
   Rows: 700,000
   Date range: 2019-01-07 23:59:00 → 2020-05-09 20:34:00

First 5 rows preview:
shape: (5, 68)
┌─────────────┬────────────┬────────────┬────────────┬───┬─────┬───────┬─────────────┬─────────────┐
│ timestamp   ┆ open       ┆ high       ┆ low        ┆ … ┆ dow ┆ month ┆ target_next ┆ target_dire │
│ ---         ┆ ---        ┆ ---        ┆ ---        ┆   ┆ --- ┆ ---   ┆ _close      ┆ ction       │
│ str         ┆ f64        ┆ f64        ┆ f64        ┆   ┆ i64 ┆ i64   ┆ ---         ┆ ---         │
│             ┆            ┆            ┆            ┆   ┆     ┆       ┆ f64         ┆ i64         │
╞═════════════╪════════════╪════════════╪════════════╪═══╪═════╪═══════╪═════════════╪═════════════╡
│ 2019-01-07  ┆ 151.520069 ┆ 151.560629 ┆ 151.42881  ┆ … ┆ 0   ┆ 1     ┆ 152.20935   ┆ 1           │
│ 23:59:00    ┆            ┆            ┆            ┆   ┆     ┆ 